# Lesson 7 — Telemetry, attribution, hard caps and circuit breakers

**Module 4 · ~12 minutes · API key required** (cheap tier on purpose)

Lessons 1–6 were *optimisation*. This lesson is why optimisation without measurement does not stick: you cannot prove the saving, you cannot stop it regressing, and you cannot kill a runaway agent. Everything runs in-process — **no Docker**. We use the floor model so the notebook stays well under a dollar.

> **Presenting:** in twenty minutes you can stand up the measurement and enforcement layer that most enterprises took six months to build. There is no technical reason to be flying blind.

### What you will be able to do

1. Tag every call with the fields that make chargeback and cost-per-task computable.
2. Watch a hard budget fire (and see the *other* team's key keep working).
3. Terminate a looping agent with three independent ceilings: steps, spend, wall-clock.
4. Report cost per **completed task**, not cost per call.

**Order of operations:** Track → Attribute → Control → Optimize. Most teams start at Optimize because it is the fun one.


### How to work through this notebook

Run cells **top to bottom**. Each section tells you what is about to happen *before* you run the code.

| Marker | What it means |
|---|---|
| **About to happen** | What the next cell will do |
| **Watch for** | The number or field that makes the point — pause on it |
| **Why it matters** | The Monday-morning decision this should change |
| **Presenting:** | Live-demo cue. Students: treat this as the takeaway |

A **cost ledger** prints at the end of every notebook that spends money.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
print(f"\nLive provider: {cfg.provider}")
print("Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.")


  Provider : none (offline)
  Arithmetic cells still run. Live cells will use rehearsal fallbacks.
  Add OPENAI_API_KEY or ANTHROPIC_API_KEY to .env for live calls.
  Rate card: verified 5 Sep 2026 — re-check before presenting.

Live provider: offline
Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.


The cell above loads `.env`, chooses **OpenAI or Anthropic** from the key you have, and prints the three model tiers this notebook will call.

**Watch for:** a banner with `Provider`, `floor`, `mid`, `frontier`.
- If it names a vendor, live cells will spend a few cents.
- If it says `offline`, arithmetic still runs. Live cells print a rehearsal fallback instead of crashing — useful on a plane, not a substitute for a real key on caching / routing / eval lessons.

Switch vendor by setting `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and re-running that cell.


---
## 1. The instrumented client

**About to happen.** We wrap `complete()` with tags and a pre-call budget check. Each call emits an OpenTelemetry-shaped span (`gen_ai.*` plus your attribution tags).

As of mid-2026 **no GenAI-specific span, metric or attribute is marked Stable**. Adopt the convention as the direction of travel and pin the version.

**Watch for:** the budget is checked *before* the call, not after. A gateway without persistent state cannot enforce a budget at all — LiteLLM is explicit: spend is read from the database.

**Normalisation trap (again):** OpenAI counts cached tokens *inside* `prompt_tokens`; Anthropic reports `cache_read_input_tokens` separately. The wrapper normalises this so the dashboard does not double-count.


In [2]:
import time, uuid, json
from collections import defaultdict

TRACES = []
BUDGETS = {}
SPEND = defaultdict(float)

class BudgetExceeded(Exception):
    pass

class CircuitBreakerTripped(Exception):
    pass

def instrumented_call(prompt, *, model=None, max_tokens=80,
                      business_unit, use_case_id, environment="prod",
                      workload_type="inference", user_id="u-000",
                      session_id=None, cost_tier=2, budget_key=None):
    """One LLM call, fully tagged, budget-enforced, emitted as an OTel-shaped span."""
    model = model or MODELS.floor
    budget_key = budget_key or business_unit

    limit = BUDGETS.get(budget_key)
    if limit is not None and SPEND[budget_key] >= limit:
        raise BudgetExceeded(
            f"ExceededBudget: key={budget_key} spend={usd(SPEND[budget_key])} limit={usd(limit)}"
        )

    t0 = time.time()
    r = complete(prompt, model=model, max_tokens=max_tokens)
    SPEND[budget_key] += r.usd

    span = {
        "gen_ai.provider.name": cfg.provider,
        "gen_ai.request.model": r.model,
        "gen_ai.usage.input_tokens": r.fresh_input + r.cache_read,
        "gen_ai.usage.output_tokens": r.output_tokens,
        "gen_ai.cache.read_tokens": r.cache_read,
        "gen_ai.cache.write_tokens": r.cache_write,
        "gen_ai.client.operation.duration": round(time.time() - t0, 3),
        "business_unit": business_unit,
        "use_case_id": use_case_id,
        "environment": environment,
        "workload_type": workload_type,
        "user_id": user_id,
        "session_id": session_id or str(uuid.uuid4())[:8],
        "cost_tier": cost_tier,
        "usd": r.usd,
    }
    TRACES.append(span)
    log_call(f"{business_unit}/{use_case_id}", r.model,
             inp=r.fresh_input, out=r.output_tokens,
             cache_w=r.cache_write, cache_r=r.cache_read)
    return r.text, span

print("instrumented client ready   cheap tier =", MODELS.floor)


instrumented client ready   cheap tier = claude-haiku-4-5


---
## 2. Simulate five use cases across three business units

**About to happen.** Ten short calls: support and classification for customer-ops, code review for engineering, RAG and summarisation for sales. Each is tagged with `business_unit`, `use_case_id`, `user_id`, `session_id`.

**Watch for:** the printed `(bu, use_case, usd)` lines. This is the raw event stream a real gateway would emit. The next cell is the dashboard nobody in most orgs has.


In [3]:
WORKLOAD = [
    ("support",        "uc-101", "customer-ops", "Summarise: shipment NW-1042 delayed 51 hours, customer claim $410."),
    ("support",        "uc-101", "customer-ops", "Summarise: shipment NW-1043 delayed 12 hours, no claim filed."),
    ("classification", "uc-102", "customer-ops", 'Classify sentiment, one word: "Third delay this month. Unacceptable."'),
    ("classification", "uc-102", "customer-ops", 'Classify sentiment, one word: "Arrived early, great service."'),
    ("code_review",    "uc-201", "engineering",  "In one sentence: what is wrong with `for i in range(len(x)): print(x[i])`?"),
    ("code_review",    "uc-201", "engineering",  "In one sentence: why prefer a context manager over manual file close?"),
    ("rag_answer",     "uc-301", "sales-mktg",   "One sentence: what is the business case for prompt caching?"),
    ("rag_answer",     "uc-301", "sales-mktg",   "One sentence: when does self-hosting an LLM beat an API?"),
    ("summarisation",  "uc-302", "sales-mktg",   "One sentence: summarise the concept of cost per completed task."),
    ("summarisation",  "uc-302", "sales-mktg",   "One sentence: summarise why output tokens cost more than input."),
]

for name, uc, bu, prompt in WORKLOAD:
    _, span = instrumented_call(prompt, business_unit=bu, use_case_id=uc,
                                user_id=f"u-{abs(hash(bu)) % 900 + 100}")
    print(f"{bu:<14} {uc:<8} {name:<15} {usd(span['usd']):>11}")
print(f"\n{len(TRACES)} spans captured")


⚠ No API key — using a rehearsal result.
customer-ops/uc-101                           $0.000419   in=19      out=80     cw=0       cr=0       
customer-ops   uc-101   support           $0.000419
⚠ No API key — using a rehearsal result.
customer-ops/uc-101                           $0.000418   in=18      out=80     cw=0       cr=0       
customer-ops   uc-101   support           $0.000418
⚠ No API key — using a rehearsal result.
customer-ops/uc-102                           $0.000416   in=16      out=80     cw=0       cr=0       
customer-ops   uc-102   classification    $0.000416
⚠ No API key — using a rehearsal result.
customer-ops/uc-102                           $0.000415   in=15      out=80     cw=0       cr=0       
customer-ops   uc-102   classification    $0.000415
⚠ No API key — using a rehearsal result.
engineering/uc-201                            $0.000422   in=22      out=80     cw=0       cr=0       
engineering    uc-201   code_review       $0.000422
⚠ No API key — using

---
## 3. The dashboard — this is Track and Attribute

**About to happen.** Three views on the same spans: spend by business unit (chargeback), spend by use case (where to aim lessons 1–6), and tag coverage (target 95%+ within 30 days).

**Watch for:** the top-3 use cases' share of spend, and any tag below 100%. Missing tags are unallocated spend — the "Compute and API usage" line finance hates.

**Why it matters.** You cannot optimize what you cannot attribute. Showback before chargeback; the political fight is easier when the numbers exist.


In [4]:
df = pd.DataFrame(TRACES)

print("=== SPEND BY BUSINESS UNIT (the chargeback view) ===")
show(df.groupby("business_unit").agg(
    calls=("usd", "size"), usd=("usd", "sum"),
    in_tok=("gen_ai.usage.input_tokens", "sum"),
    out_tok=("gen_ai.usage.output_tokens", "sum")).sort_values("usd", ascending=False)
    .style.format({"usd": "${:,.6f}"}))

print("\n=== SPEND BY USE CASE (where to aim optimisation) ===")
top = df.groupby("use_case_id").usd.sum().sort_values(ascending=False)
show(top.to_frame().style.format({"usd": "${:,.6f}"}))
print(f"Top 3 use cases carry {top.head(3).sum() / top.sum():.0%} of spend "
      "— that is where Modules 1-3 get pointed.")

print("\n=== TAG COVERAGE (target: 95%+ within 30 days) ===")
required = ["business_unit", "use_case_id", "environment", "gen_ai.request.model",
            "workload_type", "user_id", "session_id", "cost_tier"]
for tag in required:
    cov = df[tag].notna().mean() if tag in df else 0.0
    print(f"  {tag:<28} {cov:>6.0%}")


=== SPEND BY BUSINESS UNIT (the chargeback view) ===


,calls,usd,in_tok,out_tok
business_unit,,,,
customer-ops,4,$0.001668,68,320
sales-mktg,4,$0.001653,53,320
engineering,2,$0.000836,36,160



=== SPEND BY USE CASE (where to aim optimisation) ===


,usd
use_case_id,
uc-101,$0.000837
uc-201,$0.000836
uc-102,$0.000831
uc-301,$0.000827
uc-302,$0.000826


Top 3 use cases carry 60% of spend — that is where Modules 1-3 get pointed.

=== TAG COVERAGE (target: 95%+ within 30 days) ===
  business_unit                  100%
  use_case_id                    100%
  environment                    100%
  gen_ai.request.model           100%
  workload_type                  100%
  user_id                        100%
  session_id                     100%
  cost_tier                      100%


---
## 4. CONTROL — the hard cap, and proof that it fires

**About to happen.** We set customer-ops a budget just above current spend, then drive eight more calls. Engineering keeps a $5 budget.

**Watch for:** an `80% ALERT`, then `HTTP 429 ExceededBudget`. Several of the eight calls should block.

**Why it matters.** A budget you have never tested firing is not a control. Uber's $1,500/tool cap is famous because it was a *reaction*. This cell is the design-time version.

LiteLLM: *"Every budget is enforced against spend read from the database."* A stateless proxy cannot do this. Governance has an architecture prerequisite.


In [5]:
BUDGETS["customer-ops"] = SPEND["customer-ops"] + 0.00015   # tiny headroom, so it trips fast
BUDGETS["engineering"]  = 5.00                              # plenty of room

print(f"customer-ops budget: {usd(BUDGETS['customer-ops'])}  (current spend {usd(SPEND['customer-ops'])})")
print(f"engineering budget:  {usd(BUDGETS['engineering'])}\n")

blocked = 0
for i in range(8):
    try:
        _, s = instrumented_call(
            f"Reply with the single word OK. ({i})",
            business_unit="customer-ops", use_case_id="uc-101",
        )
        pct = SPEND["customer-ops"] / BUDGETS["customer-ops"]
        flag = " <-- 80% ALERT" if pct >= 0.8 else ""
        print(f"  call {i}: OK  spend={usd(SPEND['customer-ops'])} ({pct:.0%} of budget){flag}")
    except BudgetExceeded as e:
        blocked += 1
        print(f"  call {i}: HTTP 429  {e}")

print(f"\n{blocked} calls blocked by the hard cap.")
print("An untested budget is not a control. You just tested yours.")


customer-ops budget: $0.001818  (current spend $0.001668)
engineering budget:  $5.00

⚠ No API key — using a rehearsal result.
customer-ops/uc-101                           $0.000410   in=10      out=80     cw=0       cr=0       
  call 0: OK  spend=$0.002078 (114% of budget) <-- 80% ALERT
  call 1: HTTP 429  ExceededBudget: key=customer-ops spend=$0.002078 limit=$0.001818
  call 2: HTTP 429  ExceededBudget: key=customer-ops spend=$0.002078 limit=$0.001818
  call 3: HTTP 429  ExceededBudget: key=customer-ops spend=$0.002078 limit=$0.001818
  call 4: HTTP 429  ExceededBudget: key=customer-ops spend=$0.002078 limit=$0.001818
  call 5: HTTP 429  ExceededBudget: key=customer-ops spend=$0.002078 limit=$0.001818
  call 6: HTTP 429  ExceededBudget: key=customer-ops spend=$0.002078 limit=$0.001818
  call 7: HTTP 429  ExceededBudget: key=customer-ops spend=$0.002078 limit=$0.001818

7 calls blocked by the hard cap.
An untested budget is not a control. You just tested yours.


### Tenant isolation — one team's overrun must not become everyone's outage

**About to happen.** We call through the engineering key while customer-ops is already over budget.

**Watch for:** `engineering key still works`. Per-key budgets are how you stop one team from taking down the platform. A single shared API key cannot do this.


In [6]:
_, s = instrumented_call("Reply with the single word OK.",
                         business_unit="engineering", use_case_id="uc-201")
print("engineering key still works. Tenant isolation holds.")
print(f"engineering spend: {usd(SPEND['engineering'])} of {usd(BUDGETS['engineering'])}")


⚠ No API key — using a rehearsal result.
engineering/uc-201                            $0.000407   in=7       out=80     cw=0       cr=0       
engineering key still works. Tenant isolation holds.
engineering spend: $0.001243 of $5.00


---
## 5. Circuit breakers — the control that would have saved Uber's budget

**About to happen.** A deliberately looping agent ("never conclude") with four independent ceilings: steps, tokens, wall-clock, spend. Forecasts describe overruns. Limits stop them.

**Watch for:** `TERMINATED -> STEP CEILING` (or spend/token/time). Then notice that no forecast would have caught this; only a limit did.

Three limits, because they catch different failures. A token cap misses a slow-tool deadlock. A wall-clock cap misses a fast infinite loop that generates cheaply but endlessly.


In [7]:
BREAKERS = dict(max_steps=6, max_tokens=4000, max_seconds=25, max_usd=0.02)

def run_agent(goal, breakers=BREAKERS):
    sid = str(uuid.uuid4())[:8]
    t0, steps, tokens, spend = time.time(), 0, 0, 0.0
    ctx = goal
    while True:
        steps += 1
        if steps > breakers["max_steps"]:
            raise CircuitBreakerTripped(f"STEP CEILING: {steps - 1} steps (session {sid})")
        if tokens > breakers["max_tokens"]:
            raise CircuitBreakerTripped(f"TOKEN CEILING: {tokens:,} tokens (session {sid})")
        if time.time() - t0 > breakers["max_seconds"]:
            raise CircuitBreakerTripped(f"WALL-CLOCK CEILING: {time.time() - t0:.0f}s (session {sid})")
        if spend > breakers["max_usd"]:
            raise CircuitBreakerTripped(f"SPEND CEILING: {usd(spend)} (session {sid})")

        _, s = instrumented_call(
            ctx + "\n\nYou have not finished. Restate the task in one sentence and stop.",
            business_unit="engineering", use_case_id="uc-999",
            session_id=sid, max_tokens=60)
        tokens += s["gen_ai.usage.input_tokens"] + s["gen_ai.usage.output_tokens"]
        spend += s["usd"]
        ctx += " " + ("CONTEXT " * 40)
        print(f"  step {steps}: tokens={tokens:>6,}  spend={usd(spend)}  elapsed={time.time() - t0:.0f}s")

try:
    run_agent("Determine the optimal refund policy. Never conclude.")
except CircuitBreakerTripped as e:
    print(f"\nTERMINATED -> {e}")
    print("Without this, the loop runs until someone reads the invoice.")
except BudgetExceeded as e:
    print(f"\nBLOCKED AT GATEWAY -> {e}")


⚠ No API key — using a rehearsal result.
engineering/uc-999                            $0.000424   in=24      out=80     cw=0       cr=0       
  step 1: tokens=   104  spend=$0.000424  elapsed=0s
⚠ No API key — using a rehearsal result.
engineering/uc-999                            $0.000505   in=105     out=80     cw=0       cr=0       
  step 2: tokens=   289  spend=$0.000929  elapsed=0s
⚠ No API key — using a rehearsal result.
engineering/uc-999                            $0.000586   in=186     out=80     cw=0       cr=0       
  step 3: tokens=   555  spend=$0.001515  elapsed=0s
⚠ No API key — using a rehearsal result.
engineering/uc-999                            $0.000667   in=267     out=80     cw=0       cr=0       
  step 4: tokens=   902  spend=$0.002182  elapsed=0s
⚠ No API key — using a rehearsal result.
engineering/uc-999                            $0.000748   in=348     out=80     cw=0       cr=0       
  step 5: tokens= 1,330  spend=$0.002930  elapsed=0s
⚠ No API key — 

---
## 6. Cost per completed task — the only unit that survives a CFO review

**About to happen.** We group spans by `session_id` — the tag from step 1. Cost per *call* cannot see a 12-step agent. Cost per *task* can. We then divide by an assumed 92% success rate (from an eval harness, not a guess).

**Watch for:** cost per call vs cost per solved task, and any session that is a statistical outlier (candidate runaway).

**Why it matters.** This is the number from lesson 6, now computed from telemetry instead of a spreadsheet. Without `session_id` you cannot produce it.


In [8]:
df = pd.DataFrame(TRACES)
by_session = df.groupby("session_id").agg(
    calls=("usd", "size"), usd=("usd", "sum"),
    tokens=("gen_ai.usage.input_tokens", "sum")).sort_values("usd", ascending=False)
show(by_session.head(10).style.format({"usd": "${:,.6f}"}))

SUCCESS_RATE = 0.92   # from your eval harness, not from a guess
cpt = df.usd.sum() / df.session_id.nunique()
print(f"\ncost per call            {usd(df.usd.mean())}")
print(f"cost per task (attempt)  {usd(cpt)}")
print(f"cost per SOLVED task     {usd(cpt / SUCCESS_RATE)}   <- the number for the CFO")

print("\nSessions with anomalously high cost (candidate runaway agents):")
if len(by_session) > 1:
    thresh = by_session.usd.mean() + 2 * by_session.usd.std()
    show(by_session[by_session.usd > thresh].style.format({"usd": "${:,.6f}"}))
else:
    print("(need more sessions to compute a z-score)")


,calls,usd,tokens
session_id,,,
df3de9f9,6,$0.003759,1359
547d029f,1,$0.000422,22
48e6ccaf,1,$0.000419,19
3e67b76f,1,$0.000418,18
ce7ab79a,1,$0.000416,16
a84395d0,1,$0.000415,15
f94211cc,1,$0.000415,15
640409f6,1,$0.000414,14
b16b5ad3,1,$0.000413,13



cost per call            $0.000485
cost per task (attempt)  $0.000672
cost per SOLVED task     $0.000730   <- the number for the CFO

Sessions with anomalously high cost (candidate runaway agents):


,calls,usd,tokens
session_id,,,
df3de9f9,6,$0.003759,1359


---
## 7. Export — hand this to your observability platform

**About to happen.** We write `spans.jsonl` next to the notebook. In production this stream goes to Langfuse, Helicone, or your OTel collector, and the budget check moves into a gateway (LiteLLM, Portkey, Kong).

**Watch for:** the reminder that an unset budget-reset window means the budget **never resets** — a bug waiting to happen.


In [9]:
out = Path("spans.jsonl")
with out.open("w") as f:
    for s in TRACES:
        f.write(json.dumps(s) + "\n")
print(f"{len(TRACES)} spans written to {out.resolve()}")
print()
print("Next step in a real deployment:")
print("  - point these at Langfuse / Helicone / your OTel collector")
print("  - move budget enforcement into a gateway (LiteLLM, Portkey, Kong AI Gateway)")
print("  - LiteLLM enforces at: global proxy, team, user, virtual key, per-model, end-user, tag")
print("  - budget reset windows: 30s / 30m / 30h / 30d. Unset = never resets (a bug waiting to happen)")


18 spans written to notebooks/spans.jsonl

Next step in a real deployment:
  - point these at Langfuse / Helicone / your OTel collector
  - move budget enforcement into a gateway (LiteLLM, Portkey, Kong AI Gateway)
  - LiteLLM enforces at: global proxy, team, user, virtual key, per-model, end-user, tag
  - budget reset windows: 30s / 30m / 30h / 30d. Unset = never resets (a bug waiting to happen)


In [10]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.008733


,label,model,input,output,cache_write,cache_read,usd,note
0,customer-ops/uc-101,claude-haiku-4-5,19,80,0,0,0.000419,
1,customer-ops/uc-101,claude-haiku-4-5,18,80,0,0,0.000418,
2,customer-ops/uc-102,claude-haiku-4-5,16,80,0,0,0.000416,
3,customer-ops/uc-102,claude-haiku-4-5,15,80,0,0,0.000415,
4,engineering/uc-201,claude-haiku-4-5,22,80,0,0,0.000422,
5,engineering/uc-201,claude-haiku-4-5,14,80,0,0,0.000414,
6,sales-mktg/uc-301,claude-haiku-4-5,12,80,0,0,0.000412,
7,sales-mktg/uc-301,claude-haiku-4-5,15,80,0,0,0.000415,
8,sales-mktg/uc-302,claude-haiku-4-5,13,80,0,0,0.000413,
9,sales-mktg/uc-302,claude-haiku-4-5,13,80,0,0,0.000413,


---
## Takeaways

- **Track → Attribute → Control → Optimize, in that order.** Most teams start at Optimize, then cannot prove the saving or stop it regressing.
- Tag from day one. Retrofitting attribution costs far more than adding it.
- Test that your cap actually fires. An untested budget is not a control.
- Three independent circuit breakers: steps, tokens/spend, wall-clock.
- Report **cost alongside quality** on the same dashboard. Dollars without task success rewards the wrong optimisation.

Teams that can see their costs reduce them before anyone asks.

**Try on Monday:** add `business_unit`, `use_case_id`, and `session_id` to every LLM call you ship this week — even if the dashboard does not exist yet. The tags are the part you cannot backfill.
